# Session 1: Linear Programming and Maximum Flow

**Course:** Optimization and Machine Learning — FH Aachen, SS 2026  
**Prof.:** Dr.-Ing. Guido Dartmann

---

## Problem Setting

We consider a directed graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ with node set $\mathcal{V} = \{1,2,3,4,5,6\}$, source $s=1$, sink $t=6$, and directed edges $(i,j)$ with capacities $u_{ij}$.

**Goal:** Find the maximum total flow from source to sink, subject to capacity and flow conservation constraints.

### LP Formulation

**Decision variables:** $f_{ij} \in \mathbb{R}$ — flow on each edge $(i,j)$, collected in $\mathbf{f} \in \mathbb{R}^m$.

$$\max_{\mathbf{f}} \quad \sum_{j:(s,j) \in \mathcal{E}} f_{sj}$$

subject to:
$$0 \leq f_{ij} \leq u_{ij} \quad \forall (i,j) \in \mathcal{E} \qquad \text{(capacity)}$$
$$\sum_{i:(i,k)\in\mathcal{E}} f_{ik} = \sum_{j:(k,j)\in\mathcal{E}} f_{kj} \quad \forall k \in \mathcal{V} \setminus \{s,t\} \qquad \text{(flow conservation)}$$

In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

## Task 1: Getting Started with cvxpy

A simple LP to verify the setup: budget allocation across two projects.

$$\max_{x_1,x_2} \; 8x_1 + 5x_2 \quad \text{s.t.} \quad x_1+x_2 \leq 100,\; x_1 \leq 60,\; x_2 \geq 20,\; x_1,x_2 \geq 0$$

Since $P_1$ yields higher benefit per unit (8 vs. 5), we expect $x_1^* = 60$, $x_2^* = 40$.

In [ ]:
x1, x2 = cp.Variable(), cp.Variable()

prob_warmup = cp.Problem(
    cp.Maximize(8*x1 + 5*x2),
    [x1 + x2 <= 100, x1 <= 60, x2 >= 20, x1 >= 0, x2 >= 0]
)
prob_warmup.solve()

print(f"Optimal value: {prob_warmup.value:.2f}")
print(f"x1 = {x1.value:.2f},  x2 = {x2.value:.2f}")

## Task 2 & 3: Maximum Flow as a Linear Program

In [ ]:
# Graph definition: (from, to, capacity)
edges = [
    (1, 2, 7), (1, 3, 6),
    (2, 3, 2), (2, 4, 5), (2, 5, 3),
    (3, 5, 6),
    (4, 6, 6), (5, 4, 3), (5, 6, 8),
]
source, sink = 1, 6
internal_nodes = [2, 3, 4, 5]
m = len(edges)

# Decision variables: one flow variable per edge
f = cp.Variable(m, nonneg=True)
capacities = np.array([u for _, _, u in edges])

# Constraints
constraints = [f <= capacities]
for node in internal_nodes:
    inflow  = cp.sum([f[i] for i, (u, v, _) in enumerate(edges) if v == node])
    outflow = cp.sum([f[i] for i, (u, v, _) in enumerate(edges) if u == node])
    constraints.append(inflow - outflow == 0)

# Objective: maximize total outflow from source
source_outflow = cp.sum([f[i] for i, (u, v, _) in enumerate(edges) if u == source])
prob = cp.Problem(cp.Maximize(source_outflow), constraints)
prob.solve()

f_opt = f.value
eps = 1e-6

print(f"Status: {prob.status}")
print(f"Maximum flow: {prob.value:.4f}")
print()
print(f"{'Edge':<10} {'Flow':>8} {'Capacity':>10} {'Saturated':>12}")
print("-" * 44)
for i, (u, v, cap) in enumerate(edges):
    sat = abs(f_opt[i] - cap) <= eps
    print(f"({u},{v}){'':4} {f_opt[i]:>8.4f} {cap:>10} {'yes' if sat else 'no':>12}")

print()
print("Flow conservation check (inflow - outflow at internal nodes):")
for node in internal_nodes:
    inflow  = sum(f_opt[i] for i, (u, v, _) in enumerate(edges) if v == node)
    outflow = sum(f_opt[i] for i, (u, v, _) in enumerate(edges) if u == node)
    balance = inflow - outflow
    print(f"  Node {node}: balance = {balance:.2e}  {'✓' if abs(balance) <= eps else '✗'}")

## Task 4: Capacity Sensitivity

We vary the capacity of edge (4,6) and observe how the maximum flow changes.

**Expected behavior:** The optimal flow is monotonically non-decreasing in capacity. It stays flat when another bottleneck is binding.

In [ ]:
def solve_max_flow(edges_with_cap):
    m = len(edges_with_cap)
    f = cp.Variable(m, nonneg=True)
    caps = np.array([u for _, _, u in edges_with_cap])
    cons = [f <= caps]
    for node in internal_nodes:
        inflow  = cp.sum([f[i] for i, (u, v, _) in enumerate(edges_with_cap) if v == node])
        outflow = cp.sum([f[i] for i, (u, v, _) in enumerate(edges_with_cap) if u == node])
        cons.append(inflow - outflow == 0)
    src_out = cp.sum([f[i] for i, (u, v, _) in enumerate(edges_with_cap) if u == source])
    return cp.Problem(cp.Maximize(src_out), cons).solve()

target_idx = next(i for i, (u,v,_) in enumerate(edges) if u==4 and v==6)
cap_range = range(1, 20)
flow_values = []

for cap in cap_range:
    mod = list(edges)
    u, v, _ = mod[target_idx]
    mod[target_idx] = (u, v, cap)
    flow_values.append(solve_max_flow(mod))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(cap_range), flow_values, marker='o', linewidth=2, label='Max flow')
ax.axvline(x=edges[target_idx][2], color='red', linestyle='--',
           label=f'Original capacity ({edges[target_idx][2]})')
ax.set_xlabel('Capacity of edge (4,6)')
ax.set_ylabel('Maximum flow')
ax.set_title('Sensitivity: Maximum Flow vs. Capacity of Edge (4,6)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

**Interpretation:** The flow increases with capacity until another bottleneck becomes binding — at that point the curve flattens regardless of further capacity increases on this edge.

## Task 5: Greedy Baseline vs. LP

A naive greedy augments flow along any available path (BFS) without backtracking.

**Key difference:** The LP encodes correctness via constraints and finds the provably optimal solution. The greedy may get stuck depending on path choice.

In [ ]:
import time
from collections import defaultdict, deque

def greedy_max_flow(edges, source, sink):
    residual = defaultdict(float)
    neighbors = defaultdict(set)
    for u, v, cap in edges:
        residual[(u, v)] += cap
        neighbors[u].add(v)

    total_flow = 0.0
    while True:
        queue = deque([(source, [source])])
        visited, path = {source}, None
        while queue and path is None:
            node, cur = queue.popleft()
            for nb in neighbors[node]:
                if nb not in visited and residual[(node, nb)] > 1e-9:
                    new = cur + [nb]
                    if nb == sink:
                        path = new; break
                    visited.add(nb)
                    queue.append((nb, new))
        if path is None:
            break
        flow = min(residual[(path[i], path[i+1])] for i in range(len(path)-1))
        for i in range(len(path)-1):
            residual[(path[i], path[i+1])] -= flow
        total_flow += flow
    return total_flow

t0 = time.perf_counter()
greedy_result = greedy_max_flow(edges, source, sink)
t_greedy = time.perf_counter() - t0

t0 = time.perf_counter()
lp_result = solve_max_flow(edges)
t_lp = time.perf_counter() - t0

print(f"{'Method':<12} {'Flow':>8} {'Runtime':>12}")
print("-" * 34)
print(f"{'Greedy':<12} {greedy_result:>8.4f} {t_greedy*1000:>10.2f} ms")
print(f"{'LP (cvxpy)':<12} {lp_result:>8.4f} {t_lp*1000:>10.2f} ms")
print()
print("The LP formulation guarantees a globally optimal solution.")
print("The greedy provides no such guarantee — its result depends on path selection.")